# AutoData — Full External Production Validation

This notebook freezes the current candidate architecture and evaluates it on the **entire `fraudTest.csv` holdout**. It does not tune features or thresholds on the external set.

Default variants: **E0 raw**, **E1 continuous**, **E2 full**.


In [ ]:
# 0) GPU check
import os, sys, json, copy, zipfile, time, math
from pathlib import Path
import numpy as np, pandas as pd
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi || true
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. In Colab choose Runtime > Change runtime type > T4/L4/A100.")


In [ ]:
# 1) Locate/upload the project ZIP
CONTENT=Path('/content')
PROJECT_ROOT=CONTENT/'transaction-data-intelligence'
if not (PROJECT_ROOT/'src').exists():
    candidates=list(CONTENT.glob('*.zip'))
    zpath=candidates[0] if len(candidates)==1 else None
    if zpath is None:
        from google.colab import files
        print('Upload the latest AutoData project ZIP')
        uploaded=files.upload()
        if not uploaded: raise RuntimeError('No project ZIP uploaded')
        zpath=CONTENT/next(iter(uploaded))
    with zipfile.ZipFile(zpath) as z: z.extractall(CONTENT)
if not (PROJECT_ROOT/'src').exists():
    matches=[p.parent for p in CONTENT.rglob('config.yaml') if (p.parent/'src').exists()]
    if len(matches)!=1: raise RuntimeError(f'Could not locate project root: {matches}')
    PROJECT_ROOT=matches[0]
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT))
print('PROJECT_ROOT =',PROJECT_ROOT)


In [ ]:
# 2) Install dependencies
!pip install -q -r requirements.txt
import torch
assert torch.cuda.is_available()
print('GPU ready:', torch.cuda.get_device_name(0))


## Controls
Point the paths below at your Drive files. For this production-validation notebook the external test is intentionally **full**.


In [ ]:
# 3) Controls
TRAIN_PATH = "/content/drive/MyDrive/creditcardfrad_dataset/fraudTrain.csv"
EXTERNAL_PATH = "/content/drive/MyDrive/creditcardfrad_dataset/fraudTest.csv"
TARGET = "is_fraud"

TRAIN_SAMPLE_ROWS = 70_000
VALIDATION_SAMPLE_ROWS = 15_000
EXTERNAL_TEST_ROWS = "full"
TRAIN_POSITIVE_SHARE = 0.10
DEV_TRAIN_FRACTION = 0.85

SEEDS = [42,123,456]
EPOCHS = 10
PATIENCE = 4
BATCH_SIZE = 256
PREDICT_BATCH_SIZE = 2048

# Frozen production-candidate comparison. Add E2_sequence/history_sequence only if desired later.
VARIANTS = ["E0_raw","E1_continuous","E2_full"]
COST_RATIOS = [10,50,100]  # missed fraud cost relative to one false alert
print({"external":"full","variants":VARIANTS,"seeds":SEEDS})


In [ ]:
# 4) Load sources; infer schema/roles from DEVELOPMENT ONLY, then add metadata IDs
from src.ingestion.loader import load_dataset
from src.ingestion.roles import detect_roles, schema_for_profiling
from src.ingestion.schema_detector import detect_schema
from src.preprocessing.levels import DataPreparer, infer_roles
from src.profiling.profiler import parse_datetime_column
from src.utils.config import ROOT, load_config
cfg=load_config()

def ensure_file(path,label):
    p=Path(path)
    if p.exists(): return p
    from google.colab import files
    print(f'Upload {label}')
    uploaded=files.upload()
    if not uploaded: raise RuntimeError(f'Missing {label}')
    return Path('/content')/next(iter(uploaded))

train_path=ensure_file(TRAIN_PATH,'fraudTrain.csv')
ext_path=ensure_file(EXTERNAL_PATH,'fraudTest.csv')
train_ds=load_dataset(train_path, metadata_dir=ROOT/cfg['project']['data_dir']/'raw'/'_metadata')
ext_ds=load_dataset(ext_path, metadata_dir=ROOT/cfg['project']['data_dir']/'raw'/'_metadata')
train_df=train_ds.df.copy(); external_df=ext_ds.df.copy()

# Check ORIGINAL source schema before adding internal metadata.
missing_ext=set(train_df.columns)-set(external_df.columns)
missing_dev=set(external_df.columns)-set(train_df.columns)
if missing_ext or missing_dev:
    raise RuntimeError(f'Schema mismatch. Missing external={missing_ext}; missing development={missing_dev}')
external_df=external_df[train_df.columns]

schema=detect_schema(train_df, train_ds.metadata.dataset_id)
roles=detect_roles(train_df, schema, target=TARGET)
ps=schema_for_profiling(schema, roles, train_df)
level_roles=infer_roles(train_df, ps, roles)
assert level_roles.target==TARGET

# Internal metadata only AFTER schema inference.
train_df['_row_id']='dev::'+pd.Series(np.arange(len(train_df)),index=train_df.index).astype(str)
external_df['_row_id']='ext::'+pd.Series(np.arange(len(external_df)),index=external_df.index).astype(str)
assert train_df['_row_id'].is_unique and external_df['_row_id'].is_unique
assert set(train_df['_row_id']).isdisjoint(set(external_df['_row_id']))
print(f'development={len(train_df):,}; external={len(external_df):,}')
print(f'target={level_roles.target}, entity={level_roles.entity}, time={level_roles.time}, amount={level_roles.amount}')


In [ ]:
# 5) Strict chronological external holdout and frozen split
dev_time=parse_datetime_column(train_df[level_roles.time],'datetime')
ext_time=parse_datetime_column(external_df[level_roles.time],'datetime')
print('Development:',dev_time.min(),'->',dev_time.max())
print('External:   ',ext_time.min(),'->',ext_time.max())
if not (dev_time.max() < ext_time.min()):
    raise RuntimeError('External source is not strictly later than development source.')
combined=pd.concat([train_df,external_df],ignore_index=True)

external_cfg=copy.deepcopy(cfg)
external_cfg['processing']['quality_fit_scope']='train'
external_cfg['processing']['strict_external_cleaning']=True
external_cfg['sampling']['train_positive_share']=TRAIN_POSITIVE_SHARE
external_cfg['sampling']['validation_positive_share']=None
external_cfg['sampling']['keep_all_test_positives']=False
cutoff=dev_time.quantile(DEV_TRAIN_FRACTION)
split_map=pd.Series(index=combined['_row_id'],dtype='object')
split_map.loc[train_df.loc[dev_time<cutoff,'_row_id']]='train'
split_map.loc[train_df.loc[dev_time>=cutoff,'_row_id']]='validation'
split_map.loc[external_df['_row_id']]='test'
sizes={'train':TRAIN_SAMPLE_ROWS,'validation':VALIDATION_SAMPLE_ROWS,'test':len(external_df)}
bounds={'method':'fixed external source holdout','development_train':f'{dev_time.min()} to before {cutoff}',
        'development_validation':f'{cutoff} to {dev_time.max()}','external_test':f'{ext_time.min()} to {ext_time.max()}',
        'external_source':ext_path.name}
def make_preparer(config):
    p=DataPreparer(combined,ps,level_roles,config,rows='full',seed=cfg['project']['seed'])
    info=p.prepare_fixed_split(split_map,sizes,positive_share={'train':TRAIN_POSITIVE_SHARE},boundaries=bounds)
    return p,info
probe,split_info=make_preparer(external_cfg)
print(json.dumps(split_info['sample'],indent=2,default=str))
assert len(probe.sample_ids.loc[probe.sample_ids['_split']=='test'])==len(external_df)
assert set(probe.sample_ids.loc[probe.sample_ids['_split']=='test','_row_id'].str[:5])=={'ext::'}


In [ ]:
# 6) Point-in-time leakage gate
probe_e2=probe.build('E2')
pit=probe_e2.info.get('point_in_time',{})
print(json.dumps(pit,indent=2,default=str))
assert pit.get('passed'), pit
for k in ['card_history','merchant_history','previous_transactions']:
    if k in pit:
        assert pit[k]['passed']
        assert pit[k].get('same_timestamp_policy')=='strictly_earlier_only'
print('PASS: E2 history is point-in-time safe.')


In [ ]:
# 7) Frozen full-holdout evaluation + operational measurements
from src.models.sanity_transformer import SanityTransformerAdapter
from sklearn.metrics import confusion_matrix, brier_score_loss

settings={'epochs':EPOCHS,'patience':PATIENCE,'batch_size':BATCH_SIZE}
specs={
 'E0_raw':('E0',None),
 'E1_continuous':('E1','continuous'),
 'E2_full':('E2',['temporal','history','sequence']),
 'E2_sequence':('E2',['sequence']),
 'E2_history_sequence':('E2',['history','sequence']),
}
rows=[]
for variant in VARIANTS:
    level,mode=specs[variant]
    vcfg=copy.deepcopy(external_cfg)
    if variant=='E1_continuous':
        vcfg['representation']['numeric_mode']='continuous'; vcfg['representation']['numeric_coarse_bins']=0
    if level=='E2': vcfg['features']['enabled_groups']=mode
    p,_=make_preparer(vcfg)
    t0=time.perf_counter(); prepared=p.build(level); prep_s=time.perf_counter()-t0
    if level=='E2': assert prepared.info.get('point_in_time',{}).get('passed')
    test=prepared.frames['test']; y=test['_target'].to_numpy(dtype=int)
    print(f'\n=== {variant}: {len(prepared.features)} features | external={len(test):,} | prep={prep_s:.1f}s ===')
    for seed in SEEDS:
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        adapter=SanityTransformerAdapter(vcfg,device='cuda')
        summary=adapter.train(prepared,{**settings,'seed':seed})
        # Validation selects threshold; external is scored without using its labels for decisions.
        val=prepared.frames['validation']
        pv=adapter.predict(val,batch_size=PREDICT_BATCH_SIZE)
        from src.evaluation.metrics import best_f1_threshold, classification_metrics
        threshold=best_f1_threshold(val['_target'],pv,val['_weight']) if val['_target'].nunique()==2 else 0.5
        # warm-up on a small slice then time full external inference
        _=adapter.predict(test.iloc[:min(2048,len(test))],batch_size=PREDICT_BATCH_SIZE)
        torch.cuda.synchronize(); t1=time.perf_counter()
        pt=adapter.predict(test,batch_size=PREDICT_BATCH_SIZE)
        torch.cuda.synchronize(); infer_s=time.perf_counter()-t1
        m=classification_metrics(test['_target'],pt,None,threshold)
        pred=(pt>=threshold).astype(int)
        tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
        row={'variant':variant,'level':level,'seed':seed,'features':len(prepared.features),
             'external_n':len(test),'external_positives':int(y.sum()),'threshold_from_validation':threshold,
             'external_pr_auc':m['pr_auc'],'external_roc_auc':m['roc_auc'],'external_precision':m['precision'],
             'external_recall':m['recall'],'external_f1':m['f1'],'external_log_loss':m['log_loss'],
             'external_brier':float(brier_score_loss(y,pt)),'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp),
             'preprocessing_seconds':prep_s,'train_seconds':summary['train_seconds'],'inference_seconds':infer_s,
             'inference_rows_per_second':len(test)/infer_s,'inference_ms_per_1k_rows':infer_s/len(test)*1e6,
             'peak_cuda_mb':torch.cuda.max_memory_allocated()/1024**2}
        for ratio in COST_RATIOS:
            row[f'cost_fn{ratio}_fp1']=int(fp)+ratio*int(fn)
            row[f'cost_fn{ratio}_fp1_per_100k']=(int(fp)+ratio*int(fn))/len(test)*100000
        rows.append(row)
        print(f" seed={seed} PR-AUC={m['pr_auc']:.6f} P={m['precision']:.4f} R={m['recall']:.4f} F1={m['f1']:.4f} "
              f"FP={fp} FN={fn} | {len(test)/infer_s:,.0f} rows/s | peak={row['peak_cuda_mb']:.0f} MB")
results_df=pd.DataFrame(rows)
display(results_df)


In [ ]:
# 8) Stability + operational summary
metric_cols=['external_pr_auc','external_roc_auc','external_precision','external_recall','external_f1','external_log_loss',
             'external_brier','fp','fn','tp','tn','train_seconds','inference_seconds','inference_rows_per_second',
             'inference_ms_per_1k_rows','peak_cuda_mb']+[f'cost_fn{r}_fp1_per_100k' for r in COST_RATIOS]
agg=results_df.groupby('variant').agg(
    seeds=('seed','nunique'),features=('features','first'),external_n=('external_n','first'),
    **{f'{m}_mean':(m,'mean') for m in metric_cols},
    **{f'{m}_std':(m,'std') for m in ['external_pr_auc','external_recall','external_f1','inference_rows_per_second']},
).reset_index().sort_values('external_pr_auc_mean',ascending=False)
display(agg)
print('Primary decision metric: full-holdout PR-AUC. Cost scenarios are sensitivity reports at the FROZEN validation threshold, not threshold tuning.')


In [ ]:
# 9) Promotion gate (descriptive; does not alter the frozen run)
wide=agg.set_index('variant')
if {'E0_raw','E1_continuous','E2_full'} <= set(wide.index):
    e0=wide.loc['E0_raw','external_pr_auc_mean']; e1=wide.loc['E1_continuous','external_pr_auc_mean']; e2=wide.loc['E2_full','external_pr_auc_mean']
    print(f'E0 raw PR-AUC:        {e0:.6f}')
    print(f'E1 continuous PR-AUC: {e1:.6f}  delta vs E0={e1-e0:+.6f}')
    print(f'E2 full PR-AUC:       {e2:.6f}  delta vs E1={e2-e1:+.6f}; vs E0={e2-e0:+.6f}')
    print('Leakage audit:', 'PASS' if pit.get('passed') else 'FAIL')


In [ ]:
# 10) Export
from datetime import datetime
out_dir=PROJECT_ROOT/'experiments'/'production_validation'; out_dir.mkdir(parents=True,exist_ok=True)
stamp=datetime.now().strftime('%Y%m%d_%H%M%S')
long_path=out_dir/f'production_full_holdout_{stamp}.csv'
summary_path=out_dir/f'production_full_holdout_summary_{stamp}.csv'
ctx_path=out_dir/f'production_full_holdout_context_{stamp}.json'
results_df.to_csv(long_path,index=False); agg.to_csv(summary_path,index=False)
context={'development_dataset':train_ds.metadata.dataset_id,'development_file':train_path.name,
         'external_dataset':ext_ds.metadata.dataset_id,'external_file':ext_path.name,
         'strict_external_evaluation':True,'external_test_rows':len(external_df),'full_external_holdout':True,
         'schema_and_roles_fit_on':'development source only','quality_fit_scope':'train','strict_external_cleaning':True,
         'threshold_source':'development validation only','history_policy':'strictly earlier only; never labels/future peers',
         'split':split_info,'point_in_time_audit':pit,'variants':VARIANTS,'seeds':SEEDS,'epochs':EPOCHS,
         'batch_size':BATCH_SIZE,'predict_batch_size':PREDICT_BATCH_SIZE,'cost_ratios':COST_RATIOS,
         'device':torch.cuda.get_device_name(0)}
ctx_path.write_text(json.dumps(context,indent=2,default=str))
print(long_path); print(summary_path); print(ctx_path)
from google.colab import files
files.download(str(long_path)); files.download(str(summary_path)); files.download(str(ctx_path))
